# 1.多轮对话聊天机器人

In [48]:
import os

from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage,
)
from langchain_core.messages import trim_messages


# =========================
# 1. 加载配置
# =========================

load_dotenv(override=True)

MODEL_NAME = "deepseek-v4-flash"
MAX_HISTORY_TOKENS = 4000
EXIT_WORDS = {"quit", "exit"}

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

if not DEEPSEEK_API_KEY:
    raise ValueError("未找到 DEEPSEEK_API_KEY，请检查 .env 文件")

if not DEEPSEEK_BASE_URL:
    raise ValueError("未找到 DEEPSEEK_BASE_URL，请检查 .env 文件")
if not TAVILY_API_KEY:
    raise ValueError("未找到 TAVILY_API_KEY，请检查 .env 文件")


# =========================
# 2. 初始化模型
# =========================

model = init_chat_model(
    model=MODEL_NAME,
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)

# 3. 系统提示词

SYSTEM_PROMPT = """
你叫小明，是一名耐心、友好的 AI 学习助手。

你的主要任务是帮助初学者学习编程、人工智能和计算机科学。

回答要求：
1. 优先使用简单、易理解的语言。
2. 遇到复杂概念时使用例子解释。
3. 代码问题先解释思路，再给代码。
4. 如果不确定答案，请明确说明，不要编造。
5. 默认使用中文回答，专业术语可以保留英文。
"""

messages = [
    SystemMessage(content=SYSTEM_PROMPT)
]

# 4. 对话循环

print("开始聊天")
print("输入 quit 或 exit 结束对话。")

round_number = 1


while True:

    print(f"\n{'=' * 10} 第 {round_number} 轮对话 {'=' * 10}")

    user_input = input("你：").strip()

    # 判断退出

    if user_input.lower() in EXIT_WORDS:
        print("会话已结束，欢迎下次再来。")
        break

    # 防止空输入

    if not user_input:
        print("请输入有效的问题。")
        continue

    # 添加用户消息

    messages.append(
        HumanMessage(content=user_input)
    )

    # 裁剪上下文

    memory_messages = trim_messages(
        messages,
        max_tokens=MAX_HISTORY_TOKENS,
        strategy="last",
        token_counter=model,
        include_system=True,
        start_on="human",
    )

    # 模型回复

    print("小明：", end="", flush=True)

    reply_content = ""
    success = False


    try:

        for chunk in model.stream(memory_messages):

            if chunk.content:

                print(
                    chunk.content,
                    end="",
                    flush=True
                )

                reply_content += chunk.content

        success = True


    except Exception as e:

        print(f"\n模型调用失败：{e}")

    # 保存 AI 消息

    if success:

        messages.append(
            AIMessage(content=reply_content)
        )

        round_number += 1


    print()

开始聊天
输入 quit 或 exit 结束对话。

========== 第 1 轮对话 ==========
会话已结束，欢迎下次再来。


In [4]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "你是一名{subject}领域的专家，请使用适合{level}的风格进行回复。默认使用{language}。"
    ),
    (
        "human",
        "{question}"
    )
])

result = prompt.invoke({
    "subject": "LangChain",
    "level": "初学者",
    "language": "中文",
    "question": "什么是 Agent？"
})

print(result)

messages=[SystemMessage(content='你是一名LangChain领域的专家,请使用适合初学者的风格进行回复。默认使用中文。', additional_kwargs={}, response_metadata={}), HumanMessage(content='什么是 Agent？', additional_kwargs={}, response_metadata={})]


In [10]:
response = model.invoke(result)
print(response.content)

好的！我们用一个特别简单的比喻来理解 Agent。

想象一下，你有一个特别聪明的朋友，但他只会坐在那里“聊天”，不会动。你问他“明天北京天气怎么样？”他只能根据自己训练时学到的东西瞎猜，或者告诉你“我帮你查不了”。

而 Agent，就是给这个聪明的朋友装上了“眼睛”、“手”和“脚”：

1.  **眼睛**：能看文件、看网页（**检索信息**）。
2.  **手**：能操作电脑、调用软件、发邮件（**调用工具**）。
3.  **大脑**：不仅能说话，还能“思考”怎么完成任务（**规划决策**）。

**用一句话总结：Agent 就是一个“会用工具来帮你完成任务”的 AI。**

---

### 在 LangChain 里，Agent 是怎么工作的？

LangChain 是一个帮你组装 AI 应用的积木盒。在它里面，Agent 就像一个“指挥官”，它不自己干活，而是指挥其他积木干活。

它的工作流程通常是这样的：

1.  **你下达任务**：比如，“帮我给客户写一封邮件，并查一下最近的市场数据”。
2.  **Agent 思考**：它想：“哦，这个任务分两步，先写邮件需要一个文档，查数据要用搜索工具。”
3.  **Agent 调用工具**：它先去调取“文档工具”起草邮件，再用“搜索工具”查数据，然后把两者结合起来。
4.  **给你最终结果**：把一封包含最新数据的邮件草稿放在你面前。

---

### 一个更具体的例子

假设你想做一个“旅行规划助手”：

-   **一个普通的 AI**：你问“规划一下北京三日游”，它会给你一段文字建议，但可能是过时的、甚至编造的景点信息。
-   **一个 Agent**：
    -   它会调用**“天气查询工具”**，查这几天北京的天气。
    -   它会调用**“地图API工具”**，查景点之间的距离和交通方式。
    -   它会打开**“酒店预订网站”**（模拟），查找有房的酒店。
    -   最后，它把所有这些信息整合起来，给你一份**实时、可执行的完整行程表**。

---

### 给初学者的三个关键点

1.  **它不“万能”**：Agent 的能力取决于你给它装了什么工具。没装“搜索工具”，它就不会搜索。
2.  **它能“犯错”**：因为要自己规划，它可能会走冤枉路。比如先查了

In [14]:
chain = prompt | model
response1 = chain.invoke({
    "area": "LangChain",
    "level": "初学者",
    "language": "中文",
    "question": "什么是 Agent？"})
print(response1.content)

不着急，我们慢慢聊。  

在聊 LangChain 之前，我们先把“Agent”这个词搞清楚。它英文原意是“代理人”、“代理商”或者“替别人办事的人”。  

你可以把它想象成你请的一个**非常聪明、会自己想办法的私人助理**。  

---

## 1. 它和普通的“聊天机器人”有什么不一样？

普通的聊天机器人（比如以前那种客服机器人）是这样的：

> 你问：“今天天气怎么样？”  
> 机器人答：“北京晴，最高气温 25 度。”  

它只会根据关键词，从它背过的答案里挑一个给你。它**不会自己找工具**，也不会**自己定计划**。  

而 **Agent** 不一样。它更像下面这样：

> 你问：“北京今天适合穿短袖吗？”  
> Agent 的思考过程是：  
> 1. 我需要查一下天气。  
> 2. 我需要查询今天的最高气温和风力。  
> 3. 天气数据显示最高 25 度，但风比较大。  
> 4. 我得出结论：**可以穿短袖，但最好带件薄外套。**  

你看，它不是直接给答案，而是**自己做决策、调用工具（查天气）、再综合判断**。这就是 Agent 的核心。

---

## 2. 一个 Agent 由哪些部分组成？

在 LangChain 的世界里，一个 Agent 里通常有几个关键角色：

- **大脑（LLM，大语言模型，比如 ChatGPT 背后的模型）**  
  它负责“思考”。它会决定“我下一步该怎么做”。

- **手和脚（Tools，工具）**  
  就是真正干活的，比如：搜索引擎、计算器、数据库查询、天气 API 等等。Agent 缺了工具，就像一个大聪明没有手，什么都干不了。

- **计划器（Plan & Execute）**  
  有时候任务很复杂，Agent 会先列一个清单：“第一步做什么，第二步做什么。” 这比直接乱试要靠谱得多。

- **记忆（Memory）**  
  它要记得你前面说了什么，才能持续对话。就像一个好助理记得你上次说过的喜好一样。

---

## 3. 用 LangChain 来理解 Agent

如果你学过一点 LangChain，你可能会见过类似的代码：

```python
from langchain.agents import initialize_agent, Tool

In [15]:
print(type(prompt))
print(type(model))
print(type(chain))

<class 'langchain_core.prompts.chat.ChatPromptTemplate'>
<class 'langchain_deepseek.chat_models.ChatDeepSeek'>
<class 'langchain_core.runnables.base.RunnableSequence'>


In [27]:
from langchain.tools import tool
@tool
def multiply(a:float, b:float)-> float:
    """计算两个小数的乘积"""
    return a*b
result = multiply.invoke({
    "a": 5,
    "b": 3
})
print(result)
print(multiply.name)
print(multiply.description)
print(multiply.args)

15.0
multiply
计算两个小数的乘积
{'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}


In [30]:
model_with_tools = model.bind_tools([multiply])
response = model_with_tools.invoke(
    "请帮我计算 125.5 乘以 32.4"
)
print(response.tool_calls)
response1 = model_with_tools.invoke("请用一句话介绍python")
print(response1.tool_calls)

[{'name': 'multiply', 'args': {'a': 125.5, 'b': 32.4}, 'id': 'call_00_bBN7yhewPDsyZAMZkfQB1907', 'type': 'tool_call'}]
[]


In [31]:
from langchain_core.messages import HumanMessage, ToolMessage
messages=[
    HumanMessage("请帮我计算 125.5 乘以 32.4")
]
response = model_with_tools.invoke(messages)
messages.append(response)
tool_call = response.tool_calls[0]
tool_result = multiply.invoke(tool_call["args"])
tool_message = ToolMessage(

    content=str(tool_result),

    tool_call_id=tool_call["id"]

)
messages.append(tool_message)
final_response = model_with_tools.invoke(messages)
print(final_response.content)



计算完成！

**125.5 × 32.4 = 4066.2**

如果您还需要其他计算，随时告诉我。


In [37]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()
SYSTEM_PROMPT = """
你叫小明，是一名耐心、友好的 AI 学习助手。

你的主要任务是帮助初学者学习编程、人工智能和计算机科学。

回答要求：
1. 优先使用简单、易理解的语言。
2. 遇到复杂概念时使用例子解释。
3. 代码问题先解释思路，再给代码。
4. 如果不确定答案，请明确说明，不要编造。
5. 默认使用中文回答，专业术语可以保留英文。
"""
agent = create_agent(
    system_prompt=SYSTEM_PROMPT,
    model = model,
    tools = [multiply],
    checkpointer = checkpointer
)
config = {
    "configurable": {
        "thread_id": "test_1"
    }
}
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "我叫Barry，请记住我的名字"
        }
    ]
}, config=config
)
print(result["messages"][-1].content)
result2 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "我叫什么？"
            }
        ]
    },
    config=config
)
print(result2["messages"][-1].content)

好的，Barry！我记住啦！😊

我是小明，你的 AI 学习助手。以后我会直接叫你 Barry。

有什么关于编程、人工智能或者计算机科学的问题，随时都可以问我哦！
你叫 **Barry**！我刚才已经记住啦。😊

有什么想学的随时告诉我！


In [42]:
from langchain.tools import tool

@tool
def calculator(operation: str, a: float, b: float) -> float:
    """
    执行两个数字之间的基础数学运算。

    支持的 operation：
    - add：加法
    - subtract：减法
    - multiply：乘法
    - divide：除法
    """

    if operation == "add":
        return a + b

    elif operation == "subtract":
        return a - b

    elif operation == "multiply":
        return a * b

    elif operation == "divide":
        if b == 0:
            raise ValueError("除数不能为 0")
        return a / b

    else:
        raise ValueError(
            "不支持的操作，请使用 add、subtract、multiply 或 divide"
        )

calculator
执行两个数字之间的基础数学运算。

支持的 operation：
- add：加法
- subtract：减法
- multiply：乘法
- divide：除法
{'operation': {'title': 'Operation', 'type': 'string'}, 'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}


In [45]:
from datetime import datetime
from langchain.tools import tool

@tool
def get_current_time() -> str:
    """
    获取当前本地日期和时间。
    当用户询问现在几点、当前日期、今天几号等时间相关问题时使用。
    """
    now = datetime.now()

    return now.strftime("%Y-%m-%d %H:%M:%S")

2026-08-10 22:18:17
get_current_time
获取当前本地日期和时间。
当用户询问现在几点、当前日期、今天几号等时间相关问题时使用。
{}


In [50]:
from langchain_tavily import TavilySearch
web_search = TavilySearch(
    max_results=3
)
result = web_search.invoke({
    "query": "latest developments in LangChain"
})


AttributeError: 'dict' object has no attribute 'content'